# Phase 13 -- Lesion-Aware Jitter + LACN-Consistency Fine-Tune (Colab A100)

**Safe-checkpointing version.** Designed so that if the Colab session disconnects mid-training, the latest epoch checkpoint is preserved in your Google Drive.

## Workflow

1. Mount Drive (do this FIRST, before any compute) -- backup target
2. Verify GPU (A100 / V100 / T4 OK; A100 recommended)
3. Clone repo + checkout `feat/phase13-lesion-aware-consistency`
4. Install dependencies
5. Copy `best_model.pth` (Phase 6 base ckpt) from Drive into the repo
6. Download ISIC training data (3 collections, ~50 GB, ~30-40 min, cached on Drive across sessions)
7. Start keepalive thread (prevents 90-min idle disconnect)
8. Run Phase 13 fine-tune (5 epochs, ~2-3 h on A100) with **per-epoch Drive backup**
9. Validate on Phase 9 sample + OOD
10. Download final ckpt + logs to local

## Safety net
- Drive backup directory: `/content/drive/MyDrive/melanoma_p13_ckpts/`
- Per-epoch checkpoint saved (filename: `epoch_<N>_best_model_laj_consistency.pth`)
- Training log copied to Drive after each epoch
- Keepalive thread keeps the runtime alive
- If the session dies after epoch N, you have the epoch-N ckpt and can resume from there

## 1. Mount Google Drive (DO THIS FIRST)

The Drive directory is where every per-epoch checkpoint will be saved. Do NOT skip this cell -- if the session dies and Drive isn't mounted, the training is lost.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_BACKUP = '/content/drive/MyDrive/melanoma_p13_ckpts'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
print('Drive backup dir:', DRIVE_BACKUP)
print('Already exists:', os.path.exists(DRIVE_BACKUP))
!ls -la "$DRIVE_BACKUP"

## 2. Verify GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

## 3. Clone repo + checkout branch

In [ ]:
%cd /content
!rm -rf melanoma-screening-cnn
!git clone https://github.com/landerban/melanoma-screening-cnn.git
%cd melanoma-screening-cnn
!git checkout feat/phase13-lesion-aware-consistency
!git log -1 --oneline

## 4. Install dependencies

Colab already ships torch + most scientific libs; add the few extras.

In [ ]:
!pip install -q opencv-python-headless captum tqdm scikit-learn
import cv2, captum, torch, sklearn
print('cv2', cv2.__version__, '| captum', captum.__version__,
      '| torch', torch.__version__, '| sklearn', sklearn.__version__)

## 5. Copy base ckpt (`best_model.pth`) from Drive

Put `best_model.pth` (Phase 6 base, ~18 MB) at `/content/drive/MyDrive/best_model.pth` on your Drive once. If it's not there yet, the cell falls back to an interactive upload.

In [ ]:
import os, shutil
DRIVE_BEST = '/content/drive/MyDrive/best_model.pth'
LOCAL_BEST = '/content/melanoma-screening-cnn/best_model.pth'
if os.path.exists(DRIVE_BEST):
    shutil.copy(DRIVE_BEST, LOCAL_BEST)
    print('Copied base ckpt from Drive')
else:
    print('Drive copy not found at', DRIVE_BEST, '-- using interactive upload')
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        shutil.move(fname, LOCAL_BEST)
!ls -la best_model.pth

## 6. Download ISIC training data (cached on Drive across sessions)

Three collections (HAM10000 c=212, SIIM-2020 c=70, BCN20000 c=249). ~50 GB, ~30-40 min on a clean session.

**Cache strategy:** copy the `training_data/` directory to/from Drive so that subsequent sessions skip the download.

In [ ]:
import os, shutil, subprocess, time
DRIVE_DATA = '/content/drive/MyDrive/melanoma_training_data'
LOCAL_DATA = '/content/melanoma-screening-cnn/training_data'

# If Drive has a cached copy, link to it (avoid disk write on Colab disk)
if os.path.exists(DRIVE_DATA) and os.path.exists(os.path.join(DRIVE_DATA, 'images')):
    if not os.path.exists(LOCAL_DATA):
        os.symlink(DRIVE_DATA, LOCAL_DATA)
    print('Using Drive-cached training_data ->', os.readlink(LOCAL_DATA) if os.path.islink(LOCAL_DATA) else LOCAL_DATA)
    !ls training_data/ | head -5
    !ls training_data/images/ | wc -l
else:
    print('No Drive cache. Downloading...')
    os.makedirs(LOCAL_DATA + '/images', exist_ok=True)
    !pip install -q isic-cli
    !cd training_data && isic image download -c 212 ./images 2>&1 | tail -5
    !cd training_data && isic image download -c 70 ./images 2>&1 | tail -5
    !cd training_data && isic image download -c 249 ./images 2>&1 | tail -5
    !cd training_data && isic metadata download -c 212 -o metadata_c212.csv
    !cd training_data && isic metadata download -c 70 -o metadata_c70.csv
    !cd training_data && isic metadata download -c 249 -o metadata_c249.csv
    print('Download done. Copying to Drive for caching...')
    !rsync -a training_data/ "$DRIVE_DATA"/
    print('Cached on Drive.')

## 7. Start keepalive thread (prevents 90-min idle disconnect)

**Run this BEFORE the training cell.** It runs a tiny computation every 60 seconds in a daemon thread so Colab doesn't decide the runtime is idle.

In [ ]:
import threading, time, sys
_keepalive_started = globals().get('_keepalive_started', False)
def _keepalive():
    counter = 0
    while True:
        counter += 1
        _ = sum(range(1000))
        if counter % 30 == 0:
            sys.stdout.write(f'[keepalive] {counter} min\n'); sys.stdout.flush()
        time.sleep(60)
if not _keepalive_started:
    threading.Thread(target=_keepalive, daemon=True).start()
    globals()['_keepalive_started'] = True
    print('Keepalive thread started')
else:
    print('Keepalive already running')

## 8. Phase 13 fine-tune (5 epochs, ~2-3 h on A100)

**Safe mode enabled:**
- `--save-every-epoch`  persists the ckpt + log after every epoch to `artifacts/phase13/`
- `--drive-backup-dir`  additionally copies every epoch ckpt to `/content/drive/MyDrive/melanoma_p13_ckpts/`

If the session dies between epochs, you have the latest epoch's ckpt in Drive and can resume.

In [ ]:
DRIVE_BACKUP = '/content/drive/MyDrive/melanoma_p13_ckpts'
!mkdir -p artifacts/phase13
!python scripts/phase13/01_finetune_laj_consistency.py \
    --base-ckpt best_model.pth \
    --out-ckpt artifacts/phase13/best_model_laj_consistency.pth \
    --log-path artifacts/phase13/01_finetune.log \
    --epochs 5 --lr 1e-5 --batch-size 96 --num-workers 4 \
    --bg-hue-strength 0.15 --lambda-consistency 0.5 --lacn-focal-weight 0.5 \
    --save-every-epoch --drive-backup-dir "$DRIVE_BACKUP" 2>&1 | tee finetune_stdout.log

## 9. Defensive backup (in case the script crashed before the last save)

In [ ]:
import os, shutil, glob
DRIVE_BACKUP = '/content/drive/MyDrive/melanoma_p13_ckpts'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
n = 0
for f in glob.glob('artifacts/phase13/*.pth'):
    shutil.copy(f, DRIVE_BACKUP + '/' + os.path.basename(f))
    n += 1
for f in glob.glob('artifacts/phase13/*.log'):
    shutil.copy(f, DRIVE_BACKUP + '/' + os.path.basename(f))
    n += 1
print(f'Copied {n} files to Drive')
!ls -la "$DRIVE_BACKUP"

## 10. Validate on Phase 9 analytical sample + OOD

**Requires** `artifacts/phase9/01_analytical_sample.csv` and the 300 sample images in `training_data/images/` (already covered by the full ISIC download in step 6).

In [ ]:
# Ensure validation prerequisites are present (PAD-UFES-20, Phase 9 lesion masks, etc.)
!ls artifacts/phase9/01_analytical_sample.csv
!ls artifacts/phase9/03_shortcut_detection.csv
!ls artifacts/phase12/02_padufes_lesion_masks/ | head -5
!python scripts/phase13/02_validate.py \
    --ckpt artifacts/phase13/best_model_laj_consistency.pth \
    --out-log artifacts/phase13/02_validate.log 2>&1 | tee validate_stdout.log

## 11. Backup validation outputs to Drive

In [ ]:
import shutil, glob, os
DRIVE_BACKUP = '/content/drive/MyDrive/melanoma_p13_ckpts'
for f in glob.glob('artifacts/phase13/*.csv') + glob.glob('artifacts/phase13/*.log'):
    shutil.copy(f, DRIVE_BACKUP + '/' + os.path.basename(f))
print('Validation outputs backed up')
!ls -la "$DRIVE_BACKUP"

## 12. Download key files to your local machine

These are what you'll commit to the branch back at the local repo.

In [ ]:
from google.colab import files
# Final ckpt
files.download('artifacts/phase13/best_model_laj_consistency.pth')
# Logs
files.download('artifacts/phase13/01_finetune.log')
files.download('artifacts/phase13/02_validate.log')
# CSV results
files.download('artifacts/phase13/02_predictions.csv')
files.download('artifacts/phase13/02_counterfactual.csv')
# OOD (if produced)
import os
if os.path.exists('artifacts/phase13/02_ood_predictions.csv'):
    files.download('artifacts/phase13/02_ood_predictions.csv')
if os.path.exists('artifacts/phase13/02_ood_lacn.csv'):
    files.download('artifacts/phase13/02_ood_lacn.csv')

## 13. If a session dies and you're resuming

Re-run cells 1-7 (mount Drive, GPU, clone, install, base ckpt, data cache via Drive, keepalive). Then check the latest epoch in Drive:

```
!ls -la /content/drive/MyDrive/melanoma_p13_ckpts/
```

Copy the latest `epoch_<N>_best_model_laj_consistency.pth` into the repo as the new base and re-run the fine-tune cell with `--epochs <remaining>` and `--base-ckpt artifacts/phase13/epoch_<N>_best_model_laj_consistency.pth`.

The script's `--save-every-epoch` will continue from there.